# 00_freeze — Bike Sharing (Capital Bikeshare, 2011–2012)

**Research question:** What drives bike-sharing demand, and how do those drivers change across
working status, time of day, season, weather, and rider type?

**Goal of this notebook:** clean both datasets once, save them, and stop changing them.

| Dataset | Raw file | One row = | Answers | Frozen file |
|---|---|---|---|---|
| Hourly | `hour.csv` | one hour | *when* does demand happen? | `data/clean.csv` |
| Daily | `day.csv` | one day | *how much* demand is there? | `data/clean_day.csv` |

`data/clean.csv` is the hourly file because it is the main dataset (17,379 rows), and it is the one
the 10,000-row sample comes from. Every later notebook reads these two clean files, never the raw ones.

**Run:** put this notebook in the same folder as `hour.csv` and `day.csv`, then *Run All*.

## 1. Load both raw files

In [1]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

HOUR_RAW = Path("hour.csv")
DAY_RAW = Path("day.csv")
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)   # creates data/ if it doesn't exist yet

hour_raw = pd.read_csv(HOUR_RAW)
day_raw = pd.read_csv(DAY_RAW)
print("hour:", hour_raw.shape)   # Readme says 17,379 rows
print("day: ", day_raw.shape)    # Readme says 731 rows

hour: (17379, 17)
day:  (734, 16)


## 2. Problems in `day.csv`

The day file was edited in Excel at some point. Two things are wrong:
* the `dteday` header was overwritten with the number `366`
* extra junk rows were added at the bottom, including an Excel `#CALC!` error

Because of the junk rows, pandas read every number column as decimals or text.

In [2]:
print(list(day_raw.columns))
day_raw.tail(5)

['instant', '366', 'season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']


,instant,366,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
729,730.0,12/30/2012,1.0,1.0,12.0,0.0,0.0,0.0,1.0,0.255833,0.231700,0.483333,0.350754,364.0,1432.0,1796
730,731.0,12/31/2012,1.0,1.0,12.0,0.0,1.0,1.0,2.0,0.215833,0.223487,0.577500,0.154846,439.0,2290.0,2729
731,NaN,NaN,NaN,366.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,#CALC!
732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,256.0,NaN
733,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.0,NaN


## 3. Check the hourly data before changing anything

These are *checks*, not changes. If any `assert` fails, the file is not what I think it is, and I should stop.

In [3]:
def check_common(df, name):
    # Checks that apply to BOTH files
    assert df.isna().sum().sum() == 0, name
    assert df.duplicated().sum() == 0, name
    assert df["instant"].is_unique, name                                  # row ID
    assert (df["casual"] + df["registered"] == df["cnt"]).all(), name     # riders add up
    assert df["season"].between(1, 4).all(), name
    assert df["yr"].isin([0, 1]).all(), name
    assert df["mnth"].between(1, 12).all(), name
    assert df["weekday"].between(0, 6).all(), name
    assert df["weathersit"].between(1, 4).all(), name
    for col in ["temp", "atemp", "hum", "windspeed"]:                     # normalized to 0-1
        assert df[col].between(0, 1).all(), (name, col)
    print(name, "checks passed.")

assert hour_raw.shape == (17379, 17)
assert hour_raw["hr"].between(0, 23).all()
check_common(hour_raw, "hour")

hour checks passed.


## 4. Repair `day.csv`, then check it

**Decision:** keep only rows that have a real `instant` ID (this drops the 3 junk rows),
rename `366` back to `dteday`, and turn the columns back into whole numbers.

In [4]:
day_fixed = day_raw.rename(columns={"366": "dteday"})

# Keep rows whose 'instant' is a real number. Junk rows have it blank.
day_fixed = day_fixed[pd.to_numeric(day_fixed["instant"], errors="coerce").notna()].copy()
print("junk rows dropped:", len(day_raw) - len(day_fixed))

# Excel also changed the date format from 2011-01-01 to 1/1/2011
day_fixed["dteday"] = pd.to_datetime(day_fixed["dteday"], format="%m/%d/%Y")

# Codes and counts are whole numbers
int_cols = ["instant", "season", "yr", "mnth", "holiday", "weekday", "workingday",
            "weathersit", "casual", "registered", "cnt"]
day_fixed[int_cols] = day_fixed[int_cols].astype(int)

assert day_fixed.shape == (731, 16)
assert day_fixed["dteday"].is_unique
check_common(day_fixed, "day")

junk rows dropped: 3
day checks passed.


## 5. Do the two files agree?

They come from the same rentals, so each day's total in `day.csv` should equal the sum of that day's
hours in `hour.csv`. If this passes, the Excel edits didn't damage any real numbers in `day.csv`.

In [5]:
hour_sums = (hour_raw.assign(dteday=pd.to_datetime(hour_raw["dteday"]))
             .groupby("dteday")[["casual", "registered", "cnt"]].sum())
compare = day_fixed.set_index("dteday")[["casual", "registered", "cnt"]]

assert compare.index.equals(hour_sums.index)   # same 731 dates
assert (compare == hour_sums).all().all()      # same totals, every day
print("All 731 daily totals match the hourly sums exactly.")

All 731 daily totals match the hourly sums exactly.


## 6. Things the checks found that need a decision

**(a) Humidity = 0.** Real air never has 0% humidity. In both files it happens only on
**2011-03-10**, which points to a broken sensor that day, not real weather.

In [6]:
hour_hum0 = hour_raw[hour_raw["hum"] == 0]
day_hum0 = day_fixed[day_fixed["hum"] == 0]
print("hour:", len(hour_hum0), "rows on", hour_hum0["dteday"].unique())
print("day: ", len(day_hum0), "row on", day_hum0["dteday"].dt.date.unique())

hour: 22 rows on <StringArray>
['2011-03-10']
Length: 1, dtype: str
day:  1 row on [datetime.date(2011, 3, 10)]


**Decision:** set those humidity values to missing (`NaN`) and keep the rows, because the rental counts
are still real. pandas `.cov()` / `.corr()` skip NaN on their own; for PCA I'll need `.dropna()` first.

**(b) Wind speed = 0.** Calm air *is* possible, so I can't prove these are errors. **Decision:** keep them.

In [7]:
print("hour:", (hour_raw["windspeed"] == 0).sum(), "rows with windspeed == 0 (kept)")
print("day: ", (day_fixed["windspeed"] == 0).sum(), "rows with windspeed == 0")

hour: 2180 rows with windspeed == 0 (kept)
day:  0 rows with windspeed == 0


**(c) Missing hours.** Two years should have 731 × 24 = 17,544 hours. The hourly file has 17,379.

In [8]:
hours_per_day = hour_raw.groupby("dteday").size()
print(731 * 24 - len(hour_raw), "hours missing in total")
print("Days with fewer than 20 hours recorded:")
print(hours_per_day[hours_per_day < 20])

165 hours missing in total
Days with fewer than 20 hours recorded:
dteday
2011-01-18    12
2011-01-26    16
2011-01-27     8
2011-02-22    18
2011-08-27    18
2011-08-28    17
2012-10-29     1
2012-10-30    11
dtype: int64


Most gaps are single late-night hours with no rentals, so the hour has no row. The big gaps are real
events: **2012-10-29/30 is Hurricane Sandy** (the system shut down), and late January 2011 had a snow storm.
**Decision:** don't invent rows. These are real low-demand events. The daily file still has those days,
with their (low) totals.

**(d) The Readme's season labels are wrong.** It says season 1 = spring. The dates say otherwise:

In [9]:
hour_raw.assign(md=hour_raw["dteday"].str[5:]).groupby("season")["md"].agg(["min", "max"])

,min,max
season,,
1,01-01,12-31
2,03-21,06-20
3,06-21,09-22
4,09-23,12-20


Season 1 runs Dec 21 – Mar 20, which is **winter**. Season 2 is spring, 3 is summer, and 4 is fall.
**Decision:** label seasons by the dates, not by the Readme.

## 7. Clean both files the same way

One function, so the hourly and daily files can never be cleaned differently by accident.

In [10]:
SEASON = {1: "winter", 2: "spring", 3: "summer", 4: "fall"}             # decision 6d
WEATHER = {1: "clear", 2: "mist", 3: "light rain/snow", 4: "heavy rain/snow"}

def clean(df):
    df = df.copy()

    # Bad humidity -> missing (decision 6a)
    df.loc[df["hum"] == 0, "hum"] = np.nan

    df["dteday"] = pd.to_datetime(df["dteday"])

    # Real units. The Readme says each weather column was divided by a max value,
    # so multiplying back gives numbers I can say out loud ("25 degrees", not "0.61").
    df["temp_c"] = df["temp"] * 41        # temperature, degrees C
    df["atemp_c"] = df["atemp"] * 50      # "feels like" temperature, degrees C
    df["hum_pct"] = df["hum"] * 100       # humidity, %
    df["wind_kmh"] = df["windspeed"] * 67 # wind speed, km/h

    # Readable labels (keep the number codes too)
    df["year"] = df["yr"].map({0: 2011, 1: 2012})
    df["season_name"] = df["season"].map(SEASON)
    df["weather_name"] = df["weathersit"].map(WEATHER)
    df["day_type"] = df["workingday"].map({1: "working", 0: "non-working"})

    return df

hour = clean(hour_raw)
hour["timestamp"] = hour["dteday"] + pd.to_timedelta(hour["hr"], unit="h")
day = clean(day_fixed)

# Sort by time and put the columns in a sensible order
front = ["instant", "dteday", "year", "yr", "mnth", "season", "season_name",
         "holiday", "weekday", "workingday", "day_type", "weathersit", "weather_name"]
back = ["temp", "atemp", "hum", "windspeed", "temp_c", "atemp_c", "hum_pct", "wind_kmh",
        "casual", "registered", "cnt"]
hour = hour.sort_values("timestamp").reset_index(drop=True)[["instant", "timestamp"] + front[1:] + ["hr"] + back]
day = day.sort_values("dteday").reset_index(drop=True)[front + back]

print("hour:", hour.shape, " day:", day.shape)

hour: (17379, 26)  day: (731, 24)


## 8. Check the clean files

In [11]:
assert len(hour) == 17379 and len(day) == 731      # no rows lost
assert hour["timestamp"].is_unique and hour["timestamp"].is_monotonic_increasing
assert day["dteday"].is_unique and day["dteday"].is_monotonic_increasing

# The only missing values are the bad humidity readings
assert hour.drop(columns=["hum", "hum_pct"]).isna().sum().sum() == 0
assert day.drop(columns=["hum", "hum_pct"]).isna().sum().sum() == 0
assert hour["hum"].isna().sum() == len(hour_hum0)
assert day["hum"].isna().sum() == len(day_hum0)

day[["temp_c", "hum_pct", "wind_kmh", "casual", "registered", "cnt"]].describe().round(1)

,temp_c,hum_pct,wind_kmh,casual,registered,cnt
count,731.0,730.0,731.0,731.0,731.0,731.0
mean,20.3,62.9,12.8,848.2,3656.2,4504.3
std,7.5,14.1,5.2,686.6,1560.3,1937.2
min,2.4,18.8,1.5,2.0,20.0,22.0
25%,13.8,52.1,9.0,315.5,2497.0,3152.0
50%,20.4,62.7,12.1,713.0,3662.0,4548.0
75%,26.9,73.0,15.6,1096.0,4776.5,5956.0
max,35.3,97.2,34.0,3410.0,6946.0,8714.0


## 9. First look at the research question

A quick sanity check that the clean files answer the question. Average rentals **per day**
come from the daily file; the **hour-by-hour** pattern comes from the hourly file.

In [12]:
print("Average rentals per day, by day type:")
print(day.groupby("day_type")["cnt"].mean().round(0), "\n")

print("Average rentals per day, by season:")
print(day.groupby("season_name")["cnt"].mean().round(0).reindex(["winter", "spring", "summer", "fall"]), "\n")

print("Average rentals per day, by weather:")
print(day.groupby("weather_name")["cnt"].mean().round(0))

Average rentals per day, by day type:
day_type
non-working    4330.0
working        4585.0
Name: cnt, dtype: float64 

Average rentals per day, by season:
season_name
winter    2604.0
spring    4992.0
summer    5644.0
fall      4728.0
Name: cnt, dtype: float64 

Average rentals per day, by weather:
weather_name
clear              4877.0
light rain/snow    1803.0
mist               4036.0
Name: cnt, dtype: float64


In [13]:
# Hourly: the busiest hour on each kind of day
by_hour = hour.groupby(["day_type", "hr"])["cnt"].mean().unstack(0).round(0)
print("Busiest hour:", by_hour.idxmax().to_dict())
by_hour

Busiest hour: {'non-working': 13, 'working': 17}


day_type,non-working,working
hr,,
0,91.0,37.0
1,70.0,17.0
2,53.0,9.0
3,26.0,5.0
4,8.0,5.0
5,9.0,25.0
6,19.0,102.0
7,43.0,291.0
8,106.0,477.0


## 10. Save — this is the freeze

Both clean files stay on my laptop only (they're in `.gitignore`).
I write with `\n` line endings so the files, and their fingerprints below, come out the same on Windows and Mac.

In [14]:
CLEAN_PATH = DATA_DIR / "clean.csv"          # hourly, the main file
CLEAN_DAY_PATH = DATA_DIR / "clean_day.csv"  # daily

hour.to_csv(CLEAN_PATH, index=False, lineterminator="\n")
day.to_csv(CLEAN_DAY_PATH, index=False, lineterminator="\n")
for p in [CLEAN_PATH, CLEAN_DAY_PATH]:
    print("Saved", p, p.stat().st_size, "bytes")

Saved data/clean.csv 2644733 bytes
Saved data/clean_day.csv 109705 bytes


## 11. Freeze receipt and 10,000-row sample

> ⚠️ **Replace these cells with the code from *The Rest of the Semester*, page 2** if it is different.
> This version records what each file is, how big it is, and a SHA-256 fingerprint that changes
> if even one value in the file changes.

In [15]:
def sha256(path):
    # A fingerprint of the file's exact bytes
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def describe(df, raw_path, clean_path):
    return {
        "source_file": str(raw_path),
        "source_sha256": sha256(raw_path),
        "clean_file": str(clean_path),
        "clean_sha256": sha256(clean_path),
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "column_names": list(df.columns),
        "missing_values": {c: int(n) for c, n in df.isna().sum().items() if n > 0},
    }

receipt = {
    "dataset": "UCI Bike Sharing (Capital Bikeshare, Washington D.C.), 2011-2012",
    "research_question": "What drives bike-sharing demand, and how do those drivers change across "
                         "working status, time of day, season, weather, and rider type?",
    "hourly": describe(hour, HOUR_RAW, CLEAN_PATH),
    "daily": describe(day, DAY_RAW, CLEAN_DAY_PATH),
    "daily_matches_hourly_sums": True,   # section 5 would have stopped the notebook otherwise
    "frozen_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "pandas_version": pd.__version__,
}
Path("freeze_receipt.json").write_text(json.dumps(receipt, indent=2))
print(json.dumps(receipt, indent=2))

{
  "dataset": "UCI Bike Sharing (Capital Bikeshare, Washington D.C.), 2011-2012",
  "research_question": "What drives bike-sharing demand, and how do those drivers change across working status, time of day, season, weather, and rider type?",
  "hourly": {
    "source_file": "hour.csv",
    "source_sha256": "e03de4ee4ef4dc376ac6e04bf829673c6269e8eba5c60fa121640fa2f829504f",
    "clean_file": "data/clean.csv",
    "clean_sha256": "cda7344c16b8e286f5222ca897755461934b9122415127428c4c94517582899d",
    "rows": 17379,
    "columns": 26,
    "column_names": [
      "instant",
      "timestamp",
      "dteday",
      "year",
      "yr",
      "mnth",
      "season",
      "season_name",
      "holiday",
      "weekday",
      "workingday",
      "day_type",
      "weathersit",
      "weather_name",
      "hr",
      "temp",
      "atemp",
      "hum",
      "windspeed",
      "temp_c",
      "atemp_c",
      "hum_pct",
      "wind_kmh",
      "casual",
      "registered",
      "cnt"
    ],


In [16]:
# 10,000 random HOURS. random_state=42 means anyone who runs this gets the SAME 10,000 rows.
# (The daily file only has 731 rows, so a 10,000-row sample has to come from the hourly file.)
sample = hour.sample(n=10_000, random_state=42).sort_values("timestamp")
SAMPLE_PATH = DATA_DIR / "sample_10k.csv"
sample.to_csv(SAMPLE_PATH, index=False, lineterminator="\n")
print("Saved", SAMPLE_PATH, sample.shape, SAMPLE_PATH.stat().st_size, "bytes")

Saved data/sample_10k.csv (10000, 26) 1521401 bytes


## 12. What gets committed

| File | GitHub? |
|---|---|
| `00_freeze.ipynb` | ✅ |
| `freeze_receipt.json` | ✅ |
| `data/sample_10k.csv` | ✅ |
| `data/clean.csv` (hourly) | ❌ laptop only |
| `data/clean_day.csv` (daily) | ❌ laptop only |

**The freeze is done.** From here on, analysis notebooks start with:
```python
hour = pd.read_csv("data/clean.csv", parse_dates=["timestamp", "dteday"])
day = pd.read_csv("data/clean_day.csv", parse_dates=["dteday"])
```
and never touch `hour.csv` or `day.csv` again.